# IMDB Review Sentiment: Text Cleaning and a Bag-of-Words Baseline

This was my introduction to NLP preprocessing. The dataset is the classic IMDB labeled review set, fifty thousand movie reviews each tagged positive or negative. The goal was not to build a state-of-the-art classifier, but to understand what happens to raw text before it ever reaches a model: stripping HTML, handling punctuation and emoticons, removing stopwords, and turning cleaned text into a numeric feature matrix with a bag-of-words representation.

The classifier at the end (Random Forest on a bag-of-words matrix) is intentionally simple. It is a baseline to sanity-check the preprocessing pipeline, not a serious modeling attempt. See the closing notes for what a real next iteration would need.


In [ ]:
!pip install beautifulsoup4


In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import bs4 as bs
import nltk
from nltk.tokenize import sent_tokenize # tokenizes sentences
import re
from nltk.stem import PorterStemmer
from nltk.tag import pos_tag
from nltk.corpus import stopwords
from nltk.corpus import wordnet


In [ ]:
import nltk
nltk.download(['stopwords', 'punkt', 'wordnet', 'averaged_perceptron_tagger'])


## Loading the data


In [ ]:
train_data = pd.read_csv('https://raw.githubusercontent.com/afo/data-x-plaksha/master/07a-tools-nlp-basics/data/labeledTrainData.tsv', header=0, \
                    delimiter="\t", quoting=3)


In [ ]:
train_data.shape


## Looking at one review up close

Before writing a general cleaning function, I wanted to see exactly what raw review text looks like, HTML tags and all, and confirm each cleaning step does what I expect on a single example.


In [ ]:
review3 = train_data['review'][2] # the review used for initial analysis
print(review3)
review3 = bs.BeautifulSoup(review3,features='lxml').text # removes HTML tags
print(review3[:1150])


In [ ]:
print(len(sent_tokenize(review3)))
sent_tokenize(review3) # doesn't really split all sentences
# Check if it does a better job if we add space after every period
review3 = review3.replace('.','. ')

print(len(sent_tokenize(review3)), end='\n\n') # number of sentences

review3 = re.sub('[^a-zA-Z ]' ,'',review3)
print(review3) # remove special characters


In [ ]:
review3 = review3.lower()

review3_words = review3.split()
print(review3_words[:10])


## Turning the exploration into a reusable cleaning function

Once I was confident each step worked on one review, I wrapped the whole pipeline into a single function so it could run across all fifty thousand reviews. One deliberate choice here: I kept emoticons rather than stripping them, since something like `:)` or `:(` can carry real sentiment signal that plain word tokens lose.


In [ ]:
from nltk.corpus import stopwords


def review_cleaner(review):
    '''
    Clean and preprocess a single review.

    1. Remove HTML tags
    2. Pull out emoticons before they get stripped by the punctuation regex
    3. Use regex to remove all special characters (only keep letters)
    4. Lowercase and tokenize / word split
    5. Remove English stopwords
    6. Rejoin to one string, with emoticons appended back at the end
    '''

    #1. Remove HTML tags
    review = bs.BeautifulSoup(review).text

    #2. Use regex to find emoticons
    emoticons = re.findall('(?::|;|=)(?:-)?(?:\)|\(|D|P)', review)

    #3. Remove punctuation
    review = re.sub("[^a-zA-Z]", " ",review)

    #4. Tokenize into words (all lower case)
    review = review.lower().split()

    #5. Remove stopwords
    eng_stopwords = set(stopwords.words("english"))
    review = [w for w in review if not w in eng_stopwords]

    #6. Join the review to one sentence
    review = ' '.join(review+emoticons)
    # add emoticons to the end

    return(review)


In [ ]:
%%time

num_reviews = len(train_data['review'])

review_clean_original = []

for i in range(0,num_reviews):
    if( (i+1)%500 == 0 ):
        # print progress
        print("Done with %d reviews" %(i+1)) 
    review_clean_original.append(review_cleaner(train_data['review'][i]))


## From cleaned text to numbers: bag-of-words

Models cannot work with raw strings, so the cleaned text needs to become a numeric matrix. Before applying this to the full dataset, I ran a tiny two-sentence example to make sure I understood what `CountVectorizer` was actually doing under the hood.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

sent1 = "cool students study cool data science"
sent2 = "to know data science study data science"

vect = CountVectorizer() #instantiate
vect2 = TfidfVectorizer()

sents = np.array([sent1,sent2])

vect.fit(sents);


In [ ]:
print('Total number of words in the vocabulary (and position in feature matrix):\n')
print(vect.vocabulary_)


In [ ]:
# Transform to get feature vectors

bag = vect.transform(sents)

bag.toarray()

# the rows correspond to the two example sentences


In [ ]:
vect.get_feature_names()


In [ ]:
# Put it in a DataFrame for interpretability

pd.DataFrame(bag.toarray(), columns=vect.get_feature_names(), index=[sent1,sent2])

# each cell is the raw term frequency: how many times a word appears in that document


## Building the feature matrix for the real dataset


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn import metrics # for confusion matrix, accuracy score etc

X_train, X_test, y_train, y_test = train_test_split(\
    review_clean_original, train_data['sentiment'], random_state=0, test_size=.2)


# CountVectorizer handles tokenization and vocabulary building for us
vectorizer = CountVectorizer(analyzer = "word",   \
                             tokenizer = None,    \
                             preprocessor = None, \
                             stop_words = None,   \
                             max_features = 5000)

vectorizer.fit(X_train)
print(vectorizer.get_feature_names()[:10])
train_bag = vectorizer.transform(X_train) #transform to a feature matrix
test_bag = vectorizer.transform(X_test)


In [ ]:
print(train_bag.toarray().shape)
print(test_bag.toarray().shape)
type(train_bag) # sparse matrix representation
print(train_bag)


## A baseline classifier: Random Forest on bag-of-words

With a working feature pipeline in hand, I wrapped the whole thing, vectorize, fit, evaluate, into a single function so I could reuse it if I wanted to try a different cleaning strategy later.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

def predict_sentiment(cleaned_reviews, y=train_data["sentiment"]):

    print("Creating the bag of words model\n")
    vectorizer = CountVectorizer(analyzer = "word",   \
                                 tokenizer = None,    \
                                 preprocessor = None, \
                                 stop_words = None,   \
                                 max_features = 2000) 

    X_train, X_test, y_train, y_test = train_test_split(\
    cleaned_reviews, y, random_state=0, test_size=.2)

    # fit_transform learns the vocabulary and encodes the training text;
    # the test set only gets transformed, never re-fit, to avoid leakage
    train_bag = vectorizer.fit_transform(X_train).toarray()
    test_bag = vectorizer.transform(X_test).toarray()

    print("Training the random forest classifier\n")
    forest = RandomForestClassifier(n_estimators = 50) 

    forest = forest.fit(train_bag, y_train)

    train_predictions = forest.predict(train_bag)
    test_predictions = forest.predict(test_bag)

    train_acc = metrics.accuracy_score(y_train, train_predictions)
    valid_acc = metrics.accuracy_score(y_test, test_predictions)
    print("The training accuracy is: ", train_acc, "\n", "The validation accuracy is: ", valid_acc)

    return(forest,vectorizer)


In [ ]:
np.random.seed(42)

print('Original Reviews')
forest1,vec1 = predict_sentiment(review_clean_original)


## Reading the result honestly

Training accuracy came out to 1.0 and validation accuracy to about 0.83. A perfect training score next to a meaningfully lower validation score is a textbook overfitting signature, an unconstrained Random Forest (no max depth, no minimum leaf size) will happily memorize a 5000-feature bag-of-words training set.

I am reporting both numbers rather than only the flattering one, because the gap between them is the more informative result here. It tells me the model is not just weak, it is a specific kind of weak: it is memorizing training text rather than learning generalizable sentiment signal, which is exactly what you would expect from a high-variance model on a high-dimensional sparse feature space with no regularization.

**Future work (not done here):** constrain the Random Forest (max_depth, min_samples_leaf) or try a linear model such as Logistic Regression, which tends to handle sparse bag-of-words features better than tree ensembles; add TF-IDF weighting as an alternative to raw counts; and compare against a naive baseline (majority-class prediction) to confirm the model is actually learning something beyond class balance.
